In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# E1 — STEP 2: LOAD SAVED SPLITS & INSPECT FEATURES
# ============================================================

import pandas as pd
import numpy as np

base_path = "/content/drive/MyDrive/Fraud_Project"

train = pd.read_csv(base_path + "/E1_train.csv")
validation = pd.read_csv(base_path + "/E1_validation.csv")
test = pd.read_csv(base_path + "/E1_test.csv")

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

print("\nAll columns:")
for i, col in enumerate(train.columns, 1):
    print(f"{i:2}. {col}")

print("\nNumeric columns:")
numeric_cols = train.select_dtypes(include=np.number).columns.tolist()

for col in numeric_cols:
    print("-", col)

Train shape: (18734, 23)
Validation shape: (6322, 23)
Test shape: (6185, 23)

All columns:
 1. ticker
 2. cik
 3. fiscal_year
 4. price
 5. price_date
 6. ddate
 7. Assets
 8. AssetsCurrent
 9. CashAndCashEquivalentsAtCarryingValue
10. CommonStockSharesIssued
11. CommonStockValue
12. IncomeTaxExpenseBenefit
13. LiabilitiesCurrent
14. OperatingIncomeLoss
15. StockholdersEquity
16. NetIncomeLoss
17. Liabilities
18. is_fraud
19. fraud_label_source
20. reporting_date
21. mda
22. url
23. mda_length

Numeric columns:
- cik
- fiscal_year
- price
- Assets
- AssetsCurrent
- CashAndCashEquivalentsAtCarryingValue
- CommonStockSharesIssued
- CommonStockValue
- IncomeTaxExpenseBenefit
- LiabilitiesCurrent
- OperatingIncomeLoss
- StockholdersEquity
- NetIncomeLoss
- Liabilities
- is_fraud
- mda_length


In [ ]:
# ============================================================
# CHECK FOR ENGINEERED FINANCIAL FEATURES
# ============================================================

keywords = [
    "ratio",
    "margin",
    "roa",
    "roe",
    "growth",
    "leverage",
    "liquidity",
    "turnover",
    "debt",
    "current"
]

possible_engineered = [
    col for col in numeric_cols
    if any(k in col.lower() for k in keywords)
]

print("Possible engineered/ratio features:")
print(possible_engineered)

Possible engineered/ratio features:
['AssetsCurrent', 'LiabilitiesCurrent']


In [ ]:
# ============================================================
# E1 — STEP 3: CREATE FINANCIAL INDICATORS
# ============================================================

import pandas as pd
import numpy as np

base_path = "/content/drive/MyDrive/Fraud_Project"

train = pd.read_csv(base_path + "/E1_train.csv")
validation = pd.read_csv(base_path + "/E1_validation.csv")
test = pd.read_csv(base_path + "/E1_test.csv")


def create_financial_features(df):

    df = df.copy()

    # --------------------------------------------------------
    # Helper: avoid division by zero
    # --------------------------------------------------------
    def safe_divide(a, b):
        return a / b.replace(0, np.nan)

    # --------------------------------------------------------
    # Liquidity indicators
    # --------------------------------------------------------

    df["current_ratio"] = safe_divide(
        df["AssetsCurrent"],
        df["LiabilitiesCurrent"]
    )

    df["cash_ratio"] = safe_divide(
        df["CashAndCashEquivalentsAtCarryingValue"],
        df["LiabilitiesCurrent"]
    )

    # --------------------------------------------------------
    # Asset structure
    # --------------------------------------------------------

    df["current_assets_ratio"] = safe_divide(
        df["AssetsCurrent"],
        df["Assets"]
    )

    # --------------------------------------------------------
    # Leverage indicators
    # --------------------------------------------------------

    df["liabilities_to_assets"] = safe_divide(
        df["Liabilities"],
        df["Assets"]
    )

    df["debt_to_equity"] = safe_divide(
        df["Liabilities"],
        df["StockholdersEquity"]
    )

    # --------------------------------------------------------
    # Profitability indicators
    # --------------------------------------------------------

    df["roa"] = safe_divide(
        df["NetIncomeLoss"],
        df["Assets"]
    )

    df["roe"] = safe_divide(
        df["NetIncomeLoss"],
        df["StockholdersEquity"]
    )

    df["operating_return_on_assets"] = safe_divide(
        df["OperatingIncomeLoss"],
        df["Assets"]
    )

    # --------------------------------------------------------
    # Replace infinite values
    # --------------------------------------------------------

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    return df


# Apply identical feature engineering to all splits
train = create_financial_features(train)
validation = create_financial_features(validation)
test = create_financial_features(test)


print("Feature engineering completed.")

print("\nNew financial features:")
new_features = [
    "current_ratio",
    "cash_ratio",
    "current_assets_ratio",
    "liabilities_to_assets",
    "debt_to_equity",
    "roa",
    "roe",
    "operating_return_on_assets"
]

for feature in new_features:
    print("-", feature)

Feature engineering completed.

New financial features:
- current_ratio
- cash_ratio
- current_assets_ratio
- liabilities_to_assets
- debt_to_equity
- roa
- roe
- operating_return_on_assets


In [ ]:
# ============================================================
# CHECK NEW FEATURES
# ============================================================

print("\nTRAIN:")
print(train[new_features].describe().T)

print("\nMissing values:")
print(train[new_features].isna().sum())


TRAIN:
                              count       mean          std           min  \
current_ratio               14495.0   7.536649   107.248743     -0.001029   
cash_ratio                  13636.0   5.391674   101.127231     -0.000722   
current_assets_ratio        14493.0   0.487939     0.277160     -0.000415   
liabilities_to_assets       14393.0  48.082988  3280.549839      0.000074   
debt_to_equity              13494.0   4.835074   280.254248  -2130.757991   
roa                         16988.0  -7.997171   363.749718 -31119.321429   
roe                         16095.0  -0.769943   324.777322 -34756.774017   
operating_return_on_assets  14320.0  -4.872558   247.683349 -26044.333333   

                                 25%       50%       75%            max  
current_ratio               1.220888  1.930373  3.320827    3731.166440  
cash_ratio                  0.172382  0.485438  1.218966    3187.638310  
current_assets_ratio        0.275552  0.464028  0.685602       7.995648  
li

In [ ]:
# ============================================================
# E1 — STEP 4: CHECK RAW FINANCIAL VARIABLES
# ============================================================

raw_financial = [
    "price",
    "Assets",
    "AssetsCurrent",
    "CashAndCashEquivalentsAtCarryingValue",
    "CommonStockSharesIssued",
    "CommonStockValue",
    "IncomeTaxExpenseBenefit",
    "LiabilitiesCurrent",
    "OperatingIncomeLoss",
    "StockholdersEquity",
    "NetIncomeLoss",
    "Liabilities"
]

print("RAW FINANCIAL FEATURES")
print("=" * 60)

for col in raw_financial:
    missing = train[col].isna().sum()
    percentage = missing / len(train) * 100

    print(
        f"{col:45} "
        f"Missing: {missing:5} "
        f"({percentage:6.2f}%)"
    )

RAW FINANCIAL FEATURES
price                                         Missing:     0 (  0.00%)
Assets                                        Missing:   145 (  0.77%)
AssetsCurrent                                 Missing:  4138 ( 22.09%)
CashAndCashEquivalentsAtCarryingValue         Missing:  1924 ( 10.27%)
CommonStockSharesIssued                       Missing:  3322 ( 17.73%)
CommonStockValue                              Missing:  3180 ( 16.97%)
IncomeTaxExpenseBenefit                       Missing:  1977 ( 10.55%)
LiabilitiesCurrent                            Missing:  4164 ( 22.23%)
OperatingIncomeLoss                           Missing:  4306 ( 22.98%)
StockholdersEquity                            Missing:  1381 (  7.37%)
NetIncomeLoss                                 Missing:  1585 (  8.46%)
Liabilities                                   Missing:  4275 ( 22.82%)


In [ ]:
# ============================================================
# Check zeros in denominator variables
# ============================================================

denominator_cols = [
    "Assets",
    "AssetsCurrent",
    "LiabilitiesCurrent",
    "StockholdersEquity"
]

print("\nZERO VALUES IN IMPORTANT DENOMINATORS")
print("=" * 60)

for col in denominator_cols:
    zeros = (train[col] == 0).sum()
    print(f"{col:30} Zero values: {zeros}")


ZERO VALUES IN IMPORTANT DENOMINATORS
Assets                         Zero values: 40
AssetsCurrent                  Zero values: 36
LiabilitiesCurrent             Zero values: 0
StockholdersEquity             Zero values: 0


In [ ]:
# ============================================================
# E1 — STEP 5: DEFINE FINAL FINANCIAL FEATURES
# ============================================================

raw_financial = [
    "price",
    "Assets",
    "AssetsCurrent",
    "CashAndCashEquivalentsAtCarryingValue",
    "CommonStockSharesIssued",
    "CommonStockValue",
    "IncomeTaxExpenseBenefit",
    "LiabilitiesCurrent",
    "OperatingIncomeLoss",
    "StockholdersEquity",
    "NetIncomeLoss",
    "Liabilities"
]

engineered_financial = [
    "current_ratio",
    "cash_ratio",
    "current_assets_ratio",
    "liabilities_to_assets",
    "debt_to_equity",
    "roa",
    "roe",
    "operating_return_on_assets"
]

financial_features = raw_financial + engineered_financial

print("FINAL E1 FINANCIAL FEATURES")
print("=" * 60)

for i, feature in enumerate(financial_features, 1):
    print(f"{i:2}. {feature}")

print("\nTotal financial features:", len(financial_features))

FINAL E1 FINANCIAL FEATURES
 1. price
 2. Assets
 3. AssetsCurrent
 4. CashAndCashEquivalentsAtCarryingValue
 5. CommonStockSharesIssued
 6. CommonStockValue
 7. IncomeTaxExpenseBenefit
 8. LiabilitiesCurrent
 9. OperatingIncomeLoss
10. StockholdersEquity
11. NetIncomeLoss
12. Liabilities
13. current_ratio
14. cash_ratio
15. current_assets_ratio
16. liabilities_to_assets
17. debt_to_equity
18. roa
19. roe
20. operating_return_on_assets

Total financial features: 20


In [ ]:
# ============================================================
# E1 — STEP 6: PREPROCESS FINANCIAL FEATURES
# ============================================================

from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Make copies
# ------------------------------------------------------------

X_train = train[financial_features].copy()
X_val   = validation[financial_features].copy()
X_test  = test[financial_features].copy()

y_train = train["is_fraud"].copy()
y_val   = validation["is_fraud"].copy()
y_test  = test["is_fraud"].copy()

# ------------------------------------------------------------
# 2. Replace infinite values with NaN
# ------------------------------------------------------------

X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val   = X_val.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)

# ------------------------------------------------------------
# 3. Median imputation
# IMPORTANT:
# Median is learned ONLY from TRAINING data
# ------------------------------------------------------------

imputer = SimpleImputer(strategy="median")

X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=financial_features,
    index=X_train.index
)

X_val_imp = pd.DataFrame(
    imputer.transform(X_val),
    columns=financial_features,
    index=X_val.index
)

X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=financial_features,
    index=X_test.index
)

# ------------------------------------------------------------
# 4. Check remaining missing values
# ------------------------------------------------------------

print("Remaining missing values:")

print("Train:", X_train_imp.isna().sum().sum())
print("Validation:", X_val_imp.isna().sum().sum())
print("Test:", X_test_imp.isna().sum().sum())

print("\nPreprocessing step completed.")

Remaining missing values:
Train: 0
Validation: 0
Test: 0

Preprocessing step completed.


In [ ]:
# ============================================================
# E1 — STEP 7: CHECK EXTREME VALUES AFTER IMPUTATION
# ============================================================

print("EXTREME VALUE CHECK")
print("=" * 80)

for col in financial_features:
    q01 = X_train_imp[col].quantile(0.01)
    q99 = X_train_imp[col].quantile(0.99)
    max_val = X_train_imp[col].max()
    min_val = X_train_imp[col].min()

    print(
        f"{col:45} "
        f"1%={q01:12.4f} | "
        f"99%={q99:12.4f} | "
        f"min={min_val:12.4f} | "
        f"max={max_val:12.4f}"
    )

EXTREME VALUE CHECK
price                                         1%=      0.0553 | 99%=   4738.8625 | min=      0.0001 | max=8564219904.0000
Assets                                        1%=  24491.5600 | 99%=220535639999.9986 | min=      0.0000 | max=3875393000000.0000
AssetsCurrent                                 1%=  13311.4900 | 99%=27096370000.0000 | min=   -232.0000 | max=184406000000.0000
CashAndCashEquivalentsAtCarryingValue         1%=   4004.6200 | 99%=8938690000.0000 | min=   -275.0000 | max=261036000000.0000
CommonStockSharesIssued                       1%=1268938.5400 | 99%=2038893979.0000 | min=      0.0000 | max=273040243000000.0000
CommonStockValue                              1%=      0.0000 | 99%=2708670000.0000 | min=-2394931.0000 | max=35129000000.0000
IncomeTaxExpenseBenefit                       1%=-263977000.0000 | 99%=2630340000.0000 | min=-21515000000.0000 | max=31051000000.0000
LiabilitiesCurrent                            1%= 165983.6300 | 99%=22154750000.00

In [ ]:
# ============================================================
# E1 — STEP 8: CALCULATE CLASS WEIGHT
# ============================================================

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Negative samples:", negative_count)
print("Positive samples:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

Negative samples: 18579
Positive samples: 155
scale_pos_weight: 119.86451612903225


In [ ]:
# ============================================================
# E1 — STEP 9: XGBOOST BASELINE
# ============================================================

# Install XGBoost if needed
!pip install -q xgboost

import xgboost as xgb
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# ------------------------------------------------------------
# Create XGBoost model
# ------------------------------------------------------------

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    objective="binary:logistic",
    eval_metric="aucpr",

    scale_pos_weight=scale_pos_weight,

    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

xgb_model.fit(
    X_train_imp,
    y_train,

    eval_set=[
        (X_train_imp, y_train),
        (X_val_imp, y_val)
    ],

    verbose=False
)

print("XGBoost training completed.")

XGBoost training completed.


In [ ]:
# ============================================================
# E1 — STEP 10: XGBOOST VALIDATION RESULTS
# ============================================================

# Probability of fraud
val_prob = xgb_model.predict_proba(X_val_imp)[:, 1]

# Default threshold
val_pred = (val_prob >= 0.5).astype(int)

# Metrics
precision = precision_score(y_val, val_pred, zero_division=0)
recall = recall_score(y_val, val_pred, zero_division=0)
f1 = f1_score(y_val, val_pred, zero_division=0)
roc_auc = roc_auc_score(y_val, val_prob)
pr_auc = average_precision_score(y_val, val_prob)

print("=" * 60)
print("XGBOOST — VALIDATION RESULTS")
print("=" * 60)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"PR-AUC    : {pr_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        val_pred,
        target_names=["Non-Fraud", "Fraud"],
        zero_division=0
    )
)

XGBOOST — VALIDATION RESULTS
Precision : 0.0000
Recall    : 0.0000
F1-score  : 0.0000
ROC-AUC   : 0.4736
PR-AUC    : 0.0087

Confusion Matrix:
[[6230   35]
 [  57    0]]

Classification Report:
              precision    recall  f1-score   support

   Non-Fraud       0.99      0.99      0.99      6265
       Fraud       0.00      0.00      0.00        57

    accuracy                           0.99      6322
   macro avg       0.50      0.50      0.50      6322
weighted avg       0.98      0.99      0.98      6322



In [ ]:
# ============================================================
# E1 — STEP 11: INSPECT XGBOOST PREDICTED PROBABILITIES
# ============================================================

val_prob_series = pd.Series(val_prob)

print("Validation probability statistics")
print("=" * 60)

print(val_prob_series.describe())

print("\nFraud cases — predicted probabilities:")
print(
    pd.Series(
        val_prob[y_val.to_numpy() == 1]
    ).describe()
)

print("\nNon-fraud cases — predicted probabilities:")
print(
    pd.Series(
        val_prob[y_val.to_numpy() == 0]
    ).describe()
)

print("\nHighest predicted fraud probabilities:")
print(
    np.sort(val_prob)[-20:][::-1]
)


Validation probability statistics
count    6322.000000
mean        0.024902
std         0.069439
min         0.000002
25%         0.000689
50%         0.003516
75%         0.015087
max         0.916775
dtype: float64

Fraud cases — predicted probabilities:
count    57.000000
mean      0.017630
std       0.038691
min       0.000019
25%       0.000289
50%       0.003535
75%       0.013580
max       0.203617
dtype: float64

Non-fraud cases — predicted probabilities:
count    6265.000000
mean        0.024968
std         0.069655
min         0.000002
25%         0.000695
50%         0.003515
75%         0.015089
max         0.916775
dtype: float64

Highest predicted fraud probabilities:
[0.91677547 0.7374698  0.73070145 0.7287413  0.72194165 0.71768415
 0.70010597 0.69701135 0.69458985 0.6607574  0.6397212  0.6169318
 0.6155282  0.5959182  0.58543533 0.5601136  0.55933523 0.55509543
 0.55066574 0.5496967 ]


In [ ]:
# ============================================================
# E1 — STEP 12: VALIDATION THRESHOLD ANALYSIS
# ============================================================

thresholds = [
    0.50,
    0.30,
    0.20,
    0.10,
    0.05,
    0.02,
    0.01,
    0.005
]

print("Threshold analysis")
print("=" * 80)

for threshold in thresholds:

    pred = (val_prob >= threshold).astype(int)

    precision = precision_score(
        y_val, pred, zero_division=0
    )

    recall = recall_score(
        y_val, pred, zero_division=0
    )

    f1 = f1_score(
        y_val, pred, zero_division=0
    )

    print(
        f"Threshold={threshold:>6} | "
        f"Predicted Fraud={pred.sum():>4} | "
        f"Precision={precision:.4f} | "
        f"Recall={recall:.4f} | "
        f"F1={f1:.4f}"
    )

Threshold analysis
Threshold=   0.5 | Predicted Fraud=  35 | Precision=0.0000 | Recall=0.0000 | F1=0.0000
Threshold=   0.3 | Predicted Fraud= 107 | Precision=0.0000 | Recall=0.0000 | F1=0.0000
Threshold=   0.2 | Predicted Fraud= 189 | Precision=0.0053 | Recall=0.0175 | F1=0.0081
Threshold=   0.1 | Predicted Fraud= 362 | Precision=0.0055 | Recall=0.0351 | F1=0.0095
Threshold=  0.05 | Predicted Fraud= 726 | Precision=0.0096 | Recall=0.1228 | F1=0.0179
Threshold=  0.02 | Predicted Fraud=1351 | Precision=0.0089 | Recall=0.2105 | F1=0.0170
Threshold=  0.01 | Predicted Fraud=1993 | Precision=0.0085 | Recall=0.2982 | F1=0.0166
Threshold= 0.005 | Predicted Fraud=2763 | Precision=0.0080 | Recall=0.3860 | F1=0.0156


In [ ]:
# ============================================================
# E1 — STEP 13: CHECK ENGINEERED RATIO EXTREMES
# ============================================================

ratio_features = [
    "current_ratio",
    "cash_ratio",
    "current_assets_ratio",
    "liabilities_to_assets",
    "debt_to_equity",
    "roa",
    "roe",
    "operating_return_on_assets"
]

print("Extreme-value counts in TRAIN")
print("=" * 70)

for col in ratio_features:

    q01 = X_train_imp[col].quantile(0.01)
    q99 = X_train_imp[col].quantile(0.99)

    extreme_count = (
        (X_train_imp[col] < q01) |
        (X_train_imp[col] > q99)
    ).sum()

    print(
        f"{col:35} "
        f"1%={q01:12.4f} | "
        f"99%={q99:12.4f} | "
        f"extreme={extreme_count}"
    )

Extreme-value counts in TRAIN
current_ratio                       1%=      0.0143 | 99%=     23.9496 | extreme=376
cash_ratio                          1%=      0.0018 | 99%=     15.8891 | extreme=376
current_assets_ratio                1%=      0.0399 | 99%=      0.9994 | extreme=376
liabilities_to_assets               1%=      0.0261 | 99%=      9.3288 | extreme=376
debt_to_equity                      1%=    -11.1104 | 99%=     21.2291 | extreme=376
roa                                 1%=     -8.9920 | 99%=      0.3110 | extreme=376
roe                                 1%=     -5.2121 | 99%=      4.5682 | extreme=376
operating_return_on_assets          1%=     -5.8690 | 99%=      0.3386 | extreme=376


In [ ]:
# ============================================================
# E1 — STEP 14: TRAINING-BASED 1%-99% WINSORIZATION
# ============================================================

# Make copies
X_train_clip = X_train_imp.copy()
X_val_clip   = X_val_imp.copy()
X_test_clip  = X_test_imp.copy()

# Store training-derived limits
clip_limits = {}

for col in financial_features:

    lower = X_train_imp[col].quantile(0.01)
    upper = X_train_imp[col].quantile(0.99)

    clip_limits[col] = {
        "lower": lower,
        "upper": upper
    }

    # Apply TRAIN limits to all three datasets
    X_train_clip[col] = X_train_imp[col].clip(
        lower=lower,
        upper=upper
    )

    X_val_clip[col] = X_val_imp[col].clip(
        lower=lower,
        upper=upper
    )

    X_test_clip[col] = X_test_imp[col].clip(
        lower=lower,
        upper=upper
    )

print("Training-based clipping completed.")

print("\nExample clipping limits:")
for col in ratio_features:
    print(
        f"{col:35} "
        f"lower={clip_limits[col]['lower']:.6f} | "
        f"upper={clip_limits[col]['upper']:.6f}"
    )

Training-based clipping completed.

Example clipping limits:
current_ratio                       lower=0.014336 | upper=23.949625
cash_ratio                          lower=0.001772 | upper=15.889078
current_assets_ratio                lower=0.039905 | upper=0.999350
liabilities_to_assets               lower=0.026074 | upper=9.328791
debt_to_equity                      lower=-11.110354 | upper=21.229130
roa                                 lower=-8.992035 | upper=0.311029
roe                                 lower=-5.212053 | upper=4.568160
operating_return_on_assets          lower=-5.869017 | upper=0.338581


In [ ]:
# ============================================================
# CHECK CLIPPED FEATURES
# ============================================================

print("\nCLIPPED RATIO RANGES")
print("=" * 70)

for col in ratio_features:

    print(
        f"{col:35} "
        f"min={X_train_clip[col].min():12.4f} | "
        f"max={X_train_clip[col].max():12.4f}"
    )


CLIPPED RATIO RANGES
current_ratio                       min=      0.0143 | max=     23.9496
cash_ratio                          min=      0.0018 | max=     15.8891
current_assets_ratio                min=      0.0399 | max=      0.9994
liabilities_to_assets               min=      0.0261 | max=      9.3288
debt_to_equity                      min=    -11.1104 | max=     21.2291
roa                                 min=     -8.9920 | max=      0.3110
roe                                 min=     -5.2121 | max=      4.5682
operating_return_on_assets          min=     -5.8690 | max=      0.3386


In [ ]:
# ============================================================
# E1 — STEP 15: XGBOOST v2 — CLIPPED FEATURES
# ============================================================

import xgboost as xgb

xgb_model_v2 = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    objective="binary:logistic",
    eval_metric="aucpr",

    scale_pos_weight=scale_pos_weight,

    random_state=42,
    n_jobs=-1
)

xgb_model_v2.fit(
    X_train_clip,
    y_train,

    eval_set=[
        (X_train_clip, y_train),
        (X_val_clip, y_val)
    ],

    verbose=False
)

print("XGBoost v2 training completed.")

XGBoost v2 training completed.


In [ ]:
# ============================================================
# E1 — STEP 16: XGBOOST v2 VALIDATION RESULTS
# ============================================================

val_prob_v2 = xgb_model_v2.predict_proba(
    X_val_clip
)[:, 1]

val_pred_v2 = (val_prob_v2 >= 0.5).astype(int)

precision_v2 = precision_score(
    y_val, val_pred_v2, zero_division=0
)

recall_v2 = recall_score(
    y_val, val_pred_v2, zero_division=0
)

f1_v2 = f1_score(
    y_val, val_pred_v2, zero_division=0
)

roc_auc_v2 = roc_auc_score(
    y_val, val_prob_v2
)

pr_auc_v2 = average_precision_score(
    y_val, val_prob_v2
)

print("=" * 60)
print("XGBOOST v2 — VALIDATION RESULTS")
print("=" * 60)

print(f"Precision : {precision_v2:.4f}")
print(f"Recall    : {recall_v2:.4f}")
print(f"F1-score  : {f1_v2:.4f}")
print(f"ROC-AUC   : {roc_auc_v2:.4f}")
print(f"PR-AUC    : {pr_auc_v2:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred_v2))

XGBOOST v2 — VALIDATION RESULTS
Precision : 0.0000
Recall    : 0.0000
F1-score  : 0.0000
ROC-AUC   : 0.4660
PR-AUC    : 0.0086

Confusion Matrix:
[[6231   34]
 [  57    0]]


In [ ]:
import joblib

joblib.dump(
    xgb_model,
    "/content/drive/MyDrive/Fraud_Project/E1_XGBoost_model.pkl"
)

joblib.dump(
    imputer,
    "/content/drive/MyDrive/Fraud_Project/E1_financial_imputer.pkl"
)

print("XGBoost model and imputer saved.")

XGBoost model and imputer saved.


In [ ]:
xgb_threshold = 0.05

y_test_prob_xgb = xgb_model.predict_proba(X_test_imp)[:, 1]

y_test_pred_xgb = (
    y_test_prob_xgb >= xgb_threshold
).astype(int)

print("E1 XGBoost — FINAL TEST RESULTS")
print("-" * 45)

print("Precision :",
      precision_score(y_test, y_test_pred_xgb, zero_division=0))

print("Recall    :",
      recall_score(y_test, y_test_pred_xgb, zero_division=0))

print("F1-score  :",
      f1_score(y_test, y_test_pred_xgb, zero_division=0))

print("ROC-AUC   :",
      roc_auc_score(y_test, y_test_prob_xgb))

print("PR-AUC    :",
      average_precision_score(y_test, y_test_prob_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_xgb))

E1 XGBoost — FINAL TEST RESULTS
---------------------------------------------
Precision : 0.017543859649122806
Recall    : 0.1875
F1-score  : 0.03208556149732621
ROC-AUC   : 0.521983744486195
PR-AUC    : 0.013647460940254943

Confusion Matrix:
[[5449  672]
 [  52   12]]
